# 12 Cross-simulation check: the TNG model on Bolshoi

The packaged model was trained on TNG300-3-Dark subhalos only. Here it predicts
the whole Bolshoi box from Bolshoi halos at the same four mean separations,
and is scored against Bolshoi's own T-web labels (`bolshoi/mask`, effective
smoothing $1\,h^{-1}\,\mathrm{Mpc}$, threshold 0.65, as for TNG). Nothing
from Bolshoi was used in training, so the whole box is a test set.

What differs between the two boxes, all at once:

- cosmology (TNG: Planck 2015; Bolshoi: WMAP-like) and box size (205 vs
  $250\,h^{-1}\,\mathrm{Mpc}$);
- cell size: 0.400 vs $0.401\,h^{-1}\,\mathrm{Mpc}$ (Bolshoi on its real $624^3$ grid; see below);
- tracers: Subfind subhalos vs BDM halos, selected the same way (the most
  massive kept to reach each mean separation).

Two ways of scaling the inputs are compared (see `deepvoid.pretrained`):
`saved` uses the TNG statistics of the matching spacing (the default),
`refit` fits them on the Bolshoi box itself.

**Grid (2026-11-25).** The Bolshoi files are $624^3$ grids padded to $640^3$
(notebook 19). `GRID = 'n624'` (default) uses the real grid: the catalog's
`bolshoi/...` names crop the files and `bolshoi/mask` holds T-web labels
recomputed on $624^3$. `GRID = 'n640-padded'` reproduces the earlier run,
which read the padded grid as $0.391\,h^{-1}\,\mathrm{Mpc}$ cells. Results go
to `cross_simulation_bolshoi_n624.json`; the earlier padded results stay in
`cross_simulation_bolshoi.json`, and section 3 compares the two.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/deepvoid
!git clone -q https://github.com/skumagai3/deepvoid.git /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!git -C /content/deepvoid log --oneline -1

In [ ]:
import json
import numpy as np
import pandas as pd
from deepvoid import Catalog
from deepvoid.evaluate import confusion_matrix, scores
from deepvoid.pretrained import Pretrained

MODEL = '/content/drive/MyDrive/deepvoid_model_release'   # from notebook 11
model = Pretrained.from_folder(MODEL)
catalog = Catalog()
GRID = 'n624'                    # 'n624' (the real grid) or 'n640-padded' (as before 2026-11-25)
PREFIX = 'bolshoi/' if GRID == 'n624' else 'bolshoi/n640-padded/'
RESULTS = ('/content/drive/MyDrive/deepvoid_runs/cross_simulation_bolshoi_n624.json' if GRID == 'n624'
           else '/content/drive/MyDrive/deepvoid_runs/cross_simulation_bolshoi.json')
truth = catalog.load(PREFIX + 'mask').data
print(GRID, truth.shape)
print(model.info['name'], 'cell size', model.info['cell_size'], 'spacings', model.spacings)

## 1. Predict and score the whole Bolshoi box

In [ ]:
KEYS = ['mcc', 'void_mcc', 'void_f1', 'wall_f1', 'filament_f1', 'halo_f1']
SLICE = 312   # x index of the slice shown in section 4
rows, cms, slices = {}, {}, {}
for s in (3, 5, 7, 10):
    vol = catalog.load(f'{PREFIX}halos/L{s}')
    for stats in ('saved', 'refit'):
        pred = model.predict(vol.data, vol.box_size, float(s), stats=stats).argmax(-1)
        cm = confusion_matrix(truth, pred)
        cms[(s, stats)] = cm
        rows[(s, stats)] = {k: scores(cm)[k] for k in KEYS}
        if stats == 'saved':
            x = model.inputs(vol.data, vol.box_size, float(s))
            slices[s] = {'input': x[SLICE, ..., -1], 'pred': pred[SLICE]}
        print(s, stats, {k: round(v, 3) for k, v in rows[(s, stats)].items()}, flush=True)
    del vol, pred
bolshoi = pd.DataFrame(rows).T
bolshoi.index.names = ['spacing', 'stats']

## 2. Against the TNG test slab

In [ ]:
tng = json.load(open(f'{MODEL}/test_scores.json'))
tng = pd.DataFrame({int(k.rsplit('L', 1)[1]): {m: v[m] for m in KEYS} for k, v in tng.items()}).T
table = pd.concat({'TNG test slab': tng,
                   'Bolshoi, saved': bolshoi.xs('saved', level='stats'),
                   'Bolshoi, refit': bolshoi.xs('refit', level='stats')}, axis=1)
print(table.round(3).to_string())

## 3. Save

Scores and confusion matrices go next to the model so they can be read later.

In [ ]:
import os
out = {f'L{s} {stats}': {'scores': rows[(s, stats)], 'confusion_matrix': cms[(s, stats)].tolist()}
       for (s, stats) in rows}
path = RESULTS
json.dump(out, open(path, 'w'), indent=1)
print('saved', path)
old_path = '/content/drive/MyDrive/deepvoid_runs/cross_simulation_bolshoi.json'
if GRID == 'n624' and os.path.exists(old_path):
    old = json.load(open(old_path))
    cmp = pd.DataFrame({k: {'padded 640^3 (before)': old[k]['scores']['mcc'], 'real 624^3 (now)': out[k]['scores']['mcc']}
                        for k in out}).T
    cmp['change'] = cmp['real 624^3 (now)'] - cmp['padded 640^3 (before)']
    print('four-class MCC, padded grid against the real grid:')
    print(cmp.round(3).to_string())

## 4. Figures

Saved as PNG in `deepvoid_runs/cross_simulation_figures/`. The scores and
confusion matrices are read from the files written above, so these cells also
work after a reconnect without predicting again; only the slices need the
predictions (section 1 keeps them, or the slice cell recomputes them).

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from deepvoid.figures import CLASS_NAMES, evaluation_figure, save_and_display

FIGS = Path('/content/drive/MyDrive/deepvoid_runs/cross_simulation_figures')
FIGS.mkdir(exist_ok=True)
SUFFIX = '' if GRID == 'n640-padded' else '_n624'
SPACINGS = (3, 5, 7, 10)
bol = json.load(open(RESULTS))
tng_raw = json.load(open(f'{MODEL}/test_scores.json'))
tng_cm = {s: np.array(tng_raw[f'tng300-3/subhalos/L{s}']['confusion_matrix']) for s in SPACINGS}
bol_cm = {(s, st): np.array(bol[f'L{s} {st}']['confusion_matrix']) for s in SPACINGS
          for st in ('saved', 'refit')}

### Scores against tracer spacing

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, key, name in zip(axes, ('mcc', 'void_mcc', 'void_f1'),
                         ('four-class MCC', 'void MCC', 'void F1')):
    ax.plot(SPACINGS, [tng_raw[f'tng300-3/subhalos/L{s}'][key] for s in SPACINGS], 'o-',
            color='k', label='TNG test slab')
    ax.plot(SPACINGS, [bol[f'L{s} saved']['scores'][key] for s in SPACINGS], 's-',
            color='#2b6cb0', label='Bolshoi, saved TNG scaling')
    ax.plot(SPACINGS, [bol[f'L{s} refit']['scores'][key] for s in SPACINGS], '^--',
            color='#dd6b20', label='Bolshoi, scaling refit on Bolshoi')
    ax.set_xlabel(r'mean tracer separation $\lambda$ ($h^{-1}$ Mpc)')
    ax.set_xticks(SPACINGS); ax.grid(alpha=0.3); ax.set_title(name)
axes[0].legend(fontsize=9)
fig.suptitle('TNG-trained model: TNG test slab vs the whole Bolshoi box')
fig.tight_layout()
save_and_display(fig, FIGS / f'scores_vs_spacing{SUFFIX}.png')

### Confusion matrices

Rows are true classes and sum to 1. The bottom row is Bolshoi minus TNG:
red where Bolshoi puts more of a true class into that predicted class.

In [ ]:
def frac(cm):
    cm = np.asarray(cm, float)
    return cm / np.maximum(cm.sum(1, keepdims=True), 1)

def draw(ax, m, cmap, vmin, vmax, fmt):
    ax.imshow(m, cmap=cmap, vmin=vmin, vmax=vmax)
    for i in range(4):
        for j in range(4):
            dark = abs(m[i, j]) > 0.6 * max(abs(vmin), abs(vmax))
            ax.text(j, i, fmt.format(m[i, j]), ha='center', va='center', fontsize=9,
                    color='white' if dark else 'black')
    ax.set_xticks(range(4), CLASS_NAMES, fontsize=8)
    ax.set_yticks(range(4), CLASS_NAMES, fontsize=8)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for c, s in enumerate(SPACINGS):
    t, b = frac(tng_cm[s]), frac(bol_cm[(s, 'saved')])
    draw(axes[0, c], t, 'Blues', 0, 1, '{:.2f}')
    draw(axes[1, c], b, 'Blues', 0, 1, '{:.2f}')
    draw(axes[2, c], b - t, 'RdBu_r', -0.05, 0.05, '{:+.3f}')
    axes[0, c].set_title(rf'$\lambda = {s}$: TNG test slab')
    axes[1, c].set_title(rf'$\lambda = {s}$: Bolshoi')
    axes[2, c].set_title(rf'$\lambda = {s}$: Bolshoi $-$ TNG')
for r in range(3):
    axes[r, 0].set_ylabel('true')
for ax in axes[2]:
    ax.set_xlabel('predicted')
fig.tight_layout()
save_and_display(fig, FIGS / f'confusion_matrices{SUFFIX}.png')

### Slices through the Bolshoi box

Input (the smoothed channel, as the network sees it), Bolshoi's T-web labels,
the prediction, misclassified cells and the confusion matrix, at each
spacing. Recomputes the predictions if section 1 was not run in this session
(a few minutes per spacing).

In [ ]:
SLICE = globals().get('SLICE', 312)
slices = globals().get('slices', {})
truth = catalog.load(PREFIX + 'mask').data
for s in SPACINGS:
    if s not in slices:
        vol = catalog.load(f'{PREFIX}halos/L{s}')
        x = model.inputs(vol.data, vol.box_size, float(s))
        pred = model.predict(vol.data, vol.box_size, float(s)).argmax(-1)
        slices[s] = {'input': x[SLICE, ..., -1], 'pred': pred[SLICE]}
        del vol, x, pred
box = 250.0
rows_fig = [{'name': f'Bolshoi L{s}', 'input': slices[s]['input'], 'truth': truth[SLICE],
             'pred': slices[s]['pred'], 'input_label': 'smoothed halos',
             'result': {**scores(bol_cm[(s, 'saved')]),
                        'confusion_matrix': bol_cm[(s, 'saved')].tolist()}}
            for s in SPACINGS]
fig = evaluation_figure(rows_fig, f'TNG-trained model on Bolshoi, slice x = {SLICE} '
                        f'({SLICE * box / truth.shape[0]:.0f} h^-1 Mpc)', box_size=box)
save_and_display(fig, FIGS / f'bolshoi_slices_x{SLICE}{SUFFIX}.png')

## Shut down

Deletes this runtime once every cell above has finished, so it stops using compute units. The results are already on Drive; anything kept only under `/content` goes with the runtime. With **Run all**, Colab stops at the first cell that fails, so after an error the runtime stays up and you can look at it. Set `SHUT_DOWN = False` to keep the runtime.

In [ ]:
SHUT_DOWN = True   # False keeps the runtime after the run
if SHUT_DOWN:
    import time
    from google.colab import drive, runtime
    drive.flush_and_unmount()   # finish writing to Drive first
    time.sleep(5)
    runtime.unassign()          # disconnects and deletes this runtime